# ☀️ Pronóstico operativo solar — cuaderno de resultados

Planta de referencia **Medellín, 500 kWp DC / 400 kW AC**, pronóstico a día
adelante emitido a las 12:00 UTC del día anterior.

> **Lea esto antes que los números.** Las cifras de este cuaderno salen de una
> planta **sintética**, no de una planta real. Sirven para mostrar el método y
> el formato del resultado, no para estimar el desempeño en un sitio concreto.
> Además, la métrica y el horizonte dependen de decisiones que todavía no se
> han tomado (Fase 1). Vea la última sección.

Este cuaderno **no entrena nada**: lee artefactos ya calculados. Corre en
segundos.

In [ ]:
# Configuración. Cambie REPO por la URL de su repositorio (ver CONFIG.md).
REPO = "https://github.com/orozcoig/pronostico-solar"

import os, sys, subprocess
from pathlib import Path

NOMBRE = REPO.rstrip("/").split("/")[-1]
if "<usuario>" in REPO and not Path("artefactos").exists():
    raise SystemExit(
        "Falta configurar REPO con la URL real del repositorio. Ver CONFIG.md."
    )
if not Path(NOMBRE).exists() and not Path("artefactos").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)

RAIZ = Path(NOMBRE) if Path(NOMBRE).exists() else Path(".")
sys.path.insert(0, str(RAIZ))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas", "numpy", "pyarrow", "matplotlib", "pvlib", "scikit-learn"],
               check=False)

import json
import pandas as pd

ART = RAIZ / "artefactos"
if not (ART / "resumen.json").exists():
    raise SystemExit(
        f"No encuentro artefactos en {ART}. El repositorio debe traerlos versionados; "
        "si no, regenérelos con banco.artefactos.generar(). Ver CONFIG.md."
    )
pred = pd.read_parquet(ART / "predicciones.parquet")
conj = pd.read_parquet(ART / "conjunto.parquet")
res  = json.loads((ART / "resumen.json").read_text("utf-8"))
print(f"{res['dias_evaluados']} días evaluados · {len(pred):,} horas · listo en segundos")

## 1. Lo que pasó: generación pronosticada contra la real

La línea café es lo que la planta produjo. La azul es el pronóstico. La banda
sombreada es el rango en el que esperamos que caiga la generación 8 de cada 10
veces.

**La banda es ancha a propósito.** No es falta de ajuste: es la incertidumbre
real de pronosticar un sitio único con un día de anticipación. Una nube mueve
el cien por ciento de una planta, no una fracción del país.

In [ ]:
from banco.artefactos import figura_pronostico
from banco.sitios import MEDELLIN_REF as sitio
figura_pronostico(pred, sitio, dias=10)

## 2. Lectura en lenguaje de negocio

Cuatro números y qué significan.

In [ ]:
n = res
print(f"""
ERROR TÍPICO
  El pronóstico se equivoca, en promedio, en {n['mae_modelo_kw']:.0f} kW por hora de sol.
  Eso es el {n['nmae_modelo']:.0%} de la generación media diurna ({n['generacion_media_diurna_kw']:.0f} kW).

CONTRA QUÉ SE COMPARA
  La mejor alternativa simple (un modelo físico, sin aprendizaje) se equivoca
  en {n['mae_mejor_linea_base_kw']:.0f} kW. La mejora es de {n['mejora_sobre_linea_base_kw']:.1f} kW.
  El margen de error de esa comparación ("la vara") es {n['vara_kw']:.1f} kW,
  así que la mejora {'SÍ' if n['supera_la_vara'] else 'NO'} es distinguible del ruido.

CUÁNTO DEL ERROR SE PUEDE COMPRAR
  Si el pronóstico del tiempo fuera perfecto, el error bajaría a {n['mae_techo_alcanzable_kw']:.0f} kW.
  Es decir: el {n['fraccion_error_importada_del_meteorologico']:.0%} del error NO viene del modelo,
  viene del proveedor meteorológico. Mejorar el modelo compite por el {1-n['fraccion_error_importada_del_meteorologico']:.0%} restante.

CUÁNTA INCERTIDUMBRE HAY QUE PLANEAR
  La banda de confianza mide {n['ancho_mediano_banda_kw']:.0f} kW de ancho ({n['ancho_banda_sobre_placa']:.0%} de la capacidad),
  y acierta el {n['cobertura_banda_10_90']:.0%} de las veces (objetivo 80 %).
""")

### Por qué el tercer número es el más importante

Si el 87 % del error viene del proveedor meteorológico, **ninguna inversión en
modelado lo toca**. La conversación útil es sobre el insumo: post-procesar el
pronóstico, mezclar fuentes, usar un ensamble. Cualquier propuesta que prometa
reducir el error sin mencionar esto está prometiendo algo que no controla.

## 2b. Qué significa esto si usted toma decisiones

Sin tecnicismos. Cuatro ideas, en orden de importancia.

### Primero: qué decisión apoya esto y cuál no

Este cuaderno **no** dice cuánto va a producir una planta que todavía no existe.
Esa pregunta —si conviene construir o comprar— se responde con otra herramienta
y no es esta.

Lo que sí responde: para una planta **que ya opera**, ¿cuánto se puede saber de
antemano sobre lo que va a generar mañana, y vale la pena pagar por saberlo?

### Segundo: la banda ancha no es un defecto, es la respuesta

Mire la gráfica de arriba. La franja sombreada es donde se espera que caiga la
generación 8 de cada 10 veces. Al mediodía va casi de cero a la capacidad
total, y eso parece que el modelo no supiera nada.

Es correcto, y es el dato más importante de todos.

Una planta solar depende de una nube. A escala de país, cuando se nubla en
Medellín sigue habiendo sol en el Caribe, y el total apenas se mueve. En **una
planta**, una sola nube apaga el cien por ciento. Pronosticar un sitio único
nunca va a parecerse a pronosticar el país, y cualquier propuesta que prometa
precisión parecida está prometiendo algo imposible.

**Qué hacer con eso:** no planee contra el número central. Planee contra la
banda. Si la decisión no tolera que la generación de mañana al mediodía esté
entre muy poco y el máximo, el pronóstico no es la solución — lo es un
contrato, una batería o un margen.

### Tercero: la mayor parte del error no se compra con tecnología

El número de arriba dice que el **87 % del error no viene del modelo, viene del
pronóstico del tiempo**.

Es como culpar al conductor de llegar tarde cuando el problema es que le dieron
mal la dirección. Si el meteorólogo dice que mañana estará despejado y amanece
nublado, ningún modelo —por sofisticado que sea— arregla eso.

**Qué hacer con eso, y es la recomendación concreta:** antes de pagar por
modelos, pregúntele a su proveedor meteorológico si puede entregarle un
**ensamble** en vez de un solo pronóstico. Un ensamble es el mismo proveedor
corriendo su modelo varias veces con pequeñas variaciones, para mostrar cuánto
duda. Medido en este banco, promediar cinco de esas corridas redujo el error
**el doble** que todo el trabajo de modelado junto. Suele costar poco o nada
porque el proveedor ya lo calcula.

### Cuarto: cómo leer la cifra de error

El pronóstico se equivoca en promedio un 29 % de lo que la planta genera en una
hora de sol. Suena mucho. Dos referencias para calibrarlo:

- **Comparado con no hacer nada.** La alternativa simple —un cálculo físico sin
  aprendizaje— se equivoca más. La mejora existe y está medida, pero es modesta:
  casi todo el valor viene de usar la física del sitio, no de la inteligencia
  artificial.
- **Comparado con lo posible.** Si el pronóstico del tiempo fuera perfecto, el
  error sería diez veces menor. Ese es el techo, y dice cuánto queda realmente
  por ganar.

### En una frase

Se puede pronosticar, se sabe con cuánta precisión, y **se sabe que la mayor
parte de lo que falta no está en venta del lado del modelo**. Quien le prometa
reducir el error a la mitad sin hablar del proveedor meteorológico, o no lo ha
medido, o no se lo está diciendo.

## 2c. ¿Cuándo se justifica pagar por un pronóstico?

Dos situaciones muy distintas, y conviene no confundirlas.

### Caso A — Usted tiene historial de su planta

La planta opera, hay medidores, y existen registros de lo que produjo.
**Este es el único caso en que se puede vender un pronóstico para esa planta.**

Se justifica si se cumplen **las tres** condiciones. Si falla una, no se
justifica todavía:

**1. Hay datos suficientes.** Como mínimo doce meses de medición hora a hora.
Y además, el archivo de los pronósticos del tiempo que recibió en su momento
—no el clima que efectivamente ocurrió—. Sin eso, cualquier modelo se entrena
con información que en la vida real no va a tener, y lo que prometa no se va a
cumplir.

**2. Hay una decisión real que depende del pronóstico.** Alguien tiene que
hacer algo distinto por saber qué va a pasar mañana: comprometer energía en un
contrato, cargar una batería, mover una carga, programar un mantenimiento. Si
nadie cambia nada, el pronóstico vale cero por preciso que sea.

**3. Equivocarse cuesta dinero, y se sabe cuánto.** Y aquí hay una trampa
frecuente en Colombia: **un autogenerador pequeño —hasta 1 MW— se liquida por
mes, no por hora.** Equivocarse a las 10 de la mañana y compensarse a las 3 de
la tarde le cuesta exactamente cero. Si ese es su caso, no compre pronóstico
para liquidar; el valor, si lo hay, está en la operación (baterías, cargas
desplazables), y eso se mide distinto.

Cumplidas las tres, la inversión sensata es por etapas: primero un
**diagnóstico** de precio fijo que mide cuánto error tiene hoy y cuánto de ese
error es comprable. Ese diagnóstico puede concluir que no vale la pena seguir,
y esa también es una respuesta valiosa — mucho más barata que descubrirlo
después de construir el motor.

### Caso B — Usted solo tiene una generación calculada o simulada

Es decir, no hay historial real: hay una estimación de cuánto *debería*
producir la planta, hecha con software de simulación.

**Con eso no se puede vender un pronóstico para esa planta.** Y la razón es
sencilla: un modelo entrenado sobre datos simulados aprende una planta que no
existe. No conoce la suciedad de **sus** paneles, el recorte de **su**
inversor, la montaña que le tapa el sol a **su** arreglo a las cinco de la
tarde, ni las paradas que **su** planta tuvo el año pasado. Son exactamente las
cosas que hacen que una planta real produzca distinto de la simulación.

Lo que sí se puede hacer con datos simulados es lo que usted está viendo en
este cuaderno: **mostrar cómo se ve el resultado y qué preguntas hay que
responder**, antes de que nadie entregue datos ni firme nada. Sirve para decidir
si vale la pena empezar, no para operar.

Y hay un uso legítimo más, aunque es otro proyecto: si la planta **todavía no
existe**, la simulación física responde cuánto va a producir en un año típico.
Esa es la pregunta de inversión, y no se responde con pronóstico ni con
aprendizaje automático.

### Resumen

| Lo que usted tiene | ¿Se justifica un pronóstico? |
|---|---|
| Planta operando + 12 meses de medición + archivo de pronósticos + una decisión que depende de ello + costo del error | **Sí.** Empezar por el diagnóstico |
| Planta operando, pero menos de 12 meses, o sin archivo de pronósticos | **Todavía no.** Primero acumular datos; cambia el cronograma, no el método |
| Planta operando, pero nadie cambia una decisión por el pronóstico | **No.** Un pronóstico que ningún sistema usa vale cero |
| Autogenerador de hasta 1 MW que se liquida por mes | **No para liquidar.** Buscar el valor en la operación, o no hacerlo |
| Solo generación simulada, sin historial real | **No.** Sirve para demostrar el método, no para pronosticar su planta |
| Planta que aún no existe | **No es pronóstico.** Es simulación física de producción esperada |

## 3. Diagnóstico: qué encontramos en los datos y en el modelo

Dos revisiones automáticas. La primera busca problemas en la **medición y la
operación** de la planta; la segunda, en el **modelo**. Ambas pueden —y deben—
concluir que no hay nada que corregir.

In [ ]:
from banco.diagnostico_modelo import diagnosticar_modelo, informe
col = "q50" if "q50" in pred.columns else "modelo_fisico"
ds = diagnosticar_modelo(pred["real"], pred[col], conj, sitio,
                         mae_con_observado=res["mae_techo_alcanzable_kw"])
print(informe(ds))

Cada hallazgo trae **qué aplicar, qué NO aplicar, y qué debería pasar si el
diagnóstico es correcto**. Esa última parte es lo que lo hace verificable: si
se aplica la técnica y no ocurre lo previsto, el diagnóstico estaba mal.

In [ ]:
# Las dos vistas que delatan problemas de geometría u obstrucción.
from banco.artefactos import figura_residuo
figura_residuo(pred, conj, sitio)

## 4. Cargar los datos de su planta

El cuaderno lee a través de un **contrato de datos** único. Cambiar de la
planta sintética a su planta es cambiar la carpeta, no el código.

Formato esperado (hora local, sin zona horaria, como lo exporta un historiador):

- `sitio.json` — capacidad, inclinación, azimut, ubicación, punto de medición
- `generacion.csv` — `instante, potencia_ac_kw`
- `clima_observado.csv` — `instante, ghi_wm2, dni_wm2, dhi_wm2, temp_aire_c, viento_ms`
- `clima_pronosticado.csv` — igual, más `emitido_en` y `horizonte_h`

In [ ]:
# from banco.fuente_csv import FuenteCSV
# from banco.agente import diagnosticar
# f = FuenteCSV("mis_datos/")
# meta = f.metadatos_sitio()
# u, v = pd.Timestamp("2023-01-01", tz="UTC"), pd.Timestamp("2025-01-01", tz="UTC")
# ahora = pd.Timestamp.now(tz="UTC")
# hallazgos = diagnosticar(f.generacion(u, v, ahora), f.clima_observado(u, v), meta)
# print([h.falla for h in hallazgos] or "sin hallazgos: no hay nada que corregir")

### Qué hace falta para que esto sea útil con sus datos

Hay un criterio de entrada, y se evalúa con números antes de cotizar nada.

In [ ]:
from banco.criterios import m1_historia_minima, m2_archivo_de_pronostico, m7_historia_vs_falla, tabla
# Ajuste estos valores a su caso:
MESES_MEDICION, RESOLUCION_MIN, MESES_ARCHIVO = 24, 5, 24
FALLA_QUE_LE_PREOCUPA = "deriva_suciedad"
print(tabla([m1_historia_minima(MESES_MEDICION, RESOLUCION_MIN),
             m2_archivo_de_pronostico(MESES_ARCHIVO),
             m7_historia_vs_falla(MESES_MEDICION, FALLA_QUE_LE_PREOCUPA)]))

## 5. Qué no sabemos todavía, y qué cambiaría

Estos resultados descansan sobre decisiones que **usted** tiene que tomar, no
nosotros. Mientras no estén tomadas, ningún número de arriba es transferible a
una propuesta.

| Pregunta | Qué supusimos | Qué cambia si la respuesta es otra |
|---|---|---|
| ¿A qué hora hay que emitir el pronóstico y para qué trámite? | 12:00 UTC del día anterior, día adelante | Si es intradiario (0–6 h) manda la persistencia, no el modelo del tiempo, y el 87 % de error importado se desploma |
| ¿Cuánto cuesta pasarse y cuánto quedarse corto? | Que cuestan lo mismo | Si son distintos, el pronóstico óptimo no es la mediana sino un cuantil. Elegir mal el cuantil cuesta más que toda la mejora de modelo lograda |
| ¿Qué cantidad se liquida? | La generación bruta a la salida del inversor | Si es la exportación (generación menos autoconsumo), el objetivo del modelo cambia entero |
| ¿Qué sistema consume el pronóstico? | Ninguno definido | Un pronóstico que ningún sistema ingiere vale cero |
| ¿Hay archivo de corridas pasadas del proveedor? | 24 meses | Sin archivo cambia el cronograma. Entrenar con reanálisis en su lugar es la fuga más cara de este proyecto |

El detalle completo está en `DEPENDENCIAS_FASE1.md`.